# 08-04 · 사후 탐색 Q1: ICR 극단값을 원본 응답과 대조

| 항목 | 내용 |
|---|---|
| 하는 일 | 주 표본에서 |ICR| 상위 20개 기업-분기에 대해 ICR·영업이익 TTM·금융비용 TTM, 4분기 추출값, 원본 손익계산서의 금융·이자·영업이익 행을 출력 |
| 읽는 파일 | `outputs/07_분석/패널_기업분기.csv`, `추출값_보고서별.csv`, 저장된 응답 |
| 만드는 파일 | `outputs/07_분석/Q1_극단값_점검.txt` |
| 인증키 | 필요 없음 (저장된 파일만 읽습니다) |
| 걸리는 시간 | 약 7초 — 2026-11-05, 이 컴퓨터에서 측정 |

> 이 노트북이 **원본**입니다. 같은 이름의 `.py`는 `scripts/노트북에서_py_만들기.py`가 여기서 자동으로 만든 파일이라 직접 고치지 않습니다. 원래 손으로 쓴 `.py`는 이 폴더의 `원본_스크립트/`에 보관했습니다.

회귀는 하지 않습니다. 극단 ICR이 데이터 오류(예: 금융비용이 거의 0)에서 오는지 **눈으로 확인**하는 점검입니다(`docs/A_사후탐색_계획_11-05.md`).

### 준비 — 불러오기와 경로

필요한 도구를 불러오고, 이 폴더 위치(`HERE`)를 잡습니다. 맨 위 따옴표 글(독스트링)은 이 스크립트의 원래 사용설명서입니다.

In [1]:
"""사후 탐색 Q1: 주 표본에서 |ICR| 상위 20개 기업-분기의 자료를 원본 응답과 대조한다. (회귀 없음, docs/A_사후탐색_계획_11-05.md)
각 관측에 대해: ICR·영업이익 TTM·금융비용 TTM, 해당 4분기의 보고서별 추출값(3개월·누적), 원본 손익계산서에서 '금융·이자·영업이익' 행을 출력한다.
실행: python -X utf8 04_Q1_극단값_점검.py
"""
import json
import re
import sys
from pathlib import Path
import pandas as pd
try:
    HERE = HERE      # .py로 실행할 때
except NameError:
    HERE = Path.cwd()                           # 노트북에서 실행할 때(노트북이 있는 폴더)
sys.path.insert(0, str(HERE.parent / "01_파일럿"))
import config  # noqa: E402
if hasattr(sys.stdout, "reconfigure"):          # 노트북의 출력 창에는 이 함수가 없다
    sys.stdout.reconfigure(encoding="utf-8")

### 상수 — 입력·출력 경로와 규칙 값

In [2]:
OUT = config.BASE / "outputs" / "07_분석"
FS_DIR = config.RAW / "fs"
CODE = config.REPORTS

#### `norm()`

In [3]:
def norm(s):
    return re.sub(r"\s+", "", s or "")

#### `main()`

In [4]:
def main():
    P = pd.read_csv(OUT / "패널_기업분기.csv", dtype={"corp_code": str})
    P["corp_code"] = P["corp_code"].str.zfill(8)
    X = pd.read_csv(OUT / "추출값_보고서별.csv", dtype={"corp_code": str})
    X["corp_code"] = X["corp_code"].str.zfill(8)
    S = P[P["표본_주"] & P["icr"].notna()].copy()
    top = S.loc[S["icr"].abs().nlargest(20).index]
    lines = []
    for r in top.itertuples():
        y, q = int(r.year), int(r.q)
        lines.append("=" * 90)
        lines.append(f"기업 {r.corp_code} {y}Q{q}  ICR={r.icr:,.1f}  영업이익TTM={r.oi_ttm:,.0f}  금융비용TTM={r.fc_ttm:,.0f}  자산={r.assets:,.0f}")
        sub = X[(X["corp_code"] == r.corp_code) & (X["year"].isin([y - 1, y]))].copy()
        sub["ord"] = sub["report"].map({"Q1": 0, "H1": 1, "Q3": 2, "FY": 3})
        sub = sub.sort_values(["year", "ord"])
        for s in sub.itertuples():
            lines.append(f"   {s.year} {s.report:2s} {s.fs}  영업이익 3개월={s.op_income_m3} 누적={s.op_income_cum} | 금융비용 3개월={s.fin_cost_m3} 누적={s.fin_cost_cum}")
        # 원본 손익계산서의 금융·이자·영업이익 행 (해당 연도 사업보고서, 없으면 3분기)
        for rep in ("FY", "Q3"):
            for fs in ("CFS", "OFS"):
                p = FS_DIR / f"{r.corp_code}_{y}_{CODE[rep]}_{fs}.json"
                if p.exists() and json.loads(p.read_text(encoding="utf-8")).get("status") == "000":
                    d = json.loads(p.read_text(encoding="utf-8"))["list"]
                    rows = [x for x in d if x.get("sj_div") in ("IS", "CIS") and re.search(r"금융|이자|영업이익|영업손", norm(x.get("account_nm")))]
                    lines.append(f"   [원본 {y} {rep} {fs} 손익계산서 일부]")
                    for x in rows[:14]:
                        lines.append(f"      {norm(x['account_nm']):26s} 당기={x.get('thstrm_amount')} 누적={x.get('thstrm_add_amount')}")
                    break
            else:
                continue
            break
    text = "\n".join(lines)
    (OUT / "Q1_극단값_점검.txt").write_text(text, encoding="utf-8")
    print(text)

### (.py에만 들어가는 부분)

노트북에서는 `__name__`이 항상 `"__main__"`이라 아래 코드를 일반 셀로 두면 열 때마다 실행됩니다. 그래서 실행하지 않는 설명 셀에 두고, 변환 스크립트가 `.py`에만 넣습니다.

```python
if __name__ == "__main__":
    main()
```

In [5]:
main()

기업 00113128 2024Q2  ICR=314,864.6  영업이익TTM=36,177,942,157  금융비용TTM=114,900  자산=305,592,830,113
   2023 Q1 OFS  영업이익 3개월=9011954650.0 누적=9011954650.0 | 금융비용 3개월=50858463.0 누적=50858463.0
   2023 H1 OFS  영업이익 3개월=9119711156.0 누적=18131665806.0 | 금융비용 3개월=1609589.0 누적=52468052.0
   2023 Q3 OFS  영업이익 3개월=9825422179.0 누적=27957087985.0 | 금융비용 3개월=0.0 누적=52468052.0
   2023 FY OFS  영업이익 3개월=35795371698.0 누적=nan | 금융비용 3개월=52468052.0 누적=nan
   2024 Q1 OFS  영업이익 3개월=10108601094.0 누적=10108601094.0 | 금융비용 3개월=114900.0 누적=114900.0
   2024 H1 OFS  영업이익 3개월=8405635171.0 누적=18514236265.0 | 금융비용 3개월=0.0 누적=114900.0
   2024 Q3 OFS  영업이익 3개월=10831742811.0 누적=29345979076.0 | 금융비용 3개월=114900.0 누적=114900.0
   [원본 2024 Q3 OFS 손익계산서 일부]
      영업이익(손실)                   당기=10831742811 누적=29345979076
      금융원가                       당기=114900 누적=114900
      금융수익                       당기=480969794 누적=1187822247
기업 00238001 2019Q1  ICR=-197,190.0  영업이익TTM=-4,918,905,762  금융비용TTM=24,945  자산=169,256,428,559
   2018 